In [ ]:
# Dependency install cell (ultimate one-shot setup for Kani + Qwen HF on SageMaker)
# Run once in a fresh kernel, then restart kernel.

import sys
import subprocess

py = sys.executable

def run(cmd):
  print(">", " ".join(cmd))
  subprocess.check_call(cmd)

# 1) Clean up known conflicting packages
run([py, "-m", "pip", "uninstall", "-y", "torchvision", "transformers", "tokenizers", "kani"])

# 2) Install a compatible set for kani HuggingEngine
run([py, "-m", "pip", "install", "--no-cache-dir", "-U",
   "kani[huggingface]==1.8.0",
   "transformers==4.51.3",
   "tokenizers==0.21.1",
   "accelerate>=0.30.0",
   "huggingface_hub>=0.30.0",
   "sentencepiece",
   "safetensors",
   "protobuf",
   "nest_asyncio",
   "python-dotenv",
   "tqdm"])

# 3) Ensure torch exists (SageMaker usually already has it)
try:
  import torch  # noqa: F401
  print("torch already available")
except Exception:
  run([py, "-m", "pip", "install", "--no-cache-dir", "-U", "torch"])

print("\nInstall complete. IMPORTANT: Restart kernel now, then run Cell 1.")

In [ ]:
# Quick test cell (run after kernel restart)

import importlib.metadata as md

print("python:", __import__("sys").executable)
for p in ["kani", "torch", "transformers", "accelerate", "tokenizers", "huggingface_hub"]:
  try:
      print(f"{p}: {md.version(p)}")
  except Exception:
      print(f"{p}: NOT INSTALLED")

from kani import Kani
from kani.engines.huggingface import HuggingEngine

print("Kani import OK")
print("HuggingEngine import OK")

In [ ]:
# Cell 1: Setup imports, paths, and Qwen2.5 Kani runtime (independent)
from pathlib import Path
import os
import re
import json
import time
import asyncio
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from kani import Kani
from kani.engines.huggingface import HuggingEngine

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Put heavy caches on NVMe (recommended in SageMaker)
NVME_CACHE_ROOT = Path('/mnt/sagemaker-nvme/formalizer_cache')
HF_HOME_DIR = NVME_CACHE_ROOT / 'hf_home'
HF_HUB_CACHE_DIR = NVME_CACHE_ROOT / 'hf_hub'
TRANSFORMERS_CACHE_DIR = NVME_CACHE_ROOT / 'transformers'
TORCH_HOME_DIR = NVME_CACHE_ROOT / 'torch'
XDG_CACHE_DIR = NVME_CACHE_ROOT / 'xdg_cache'

for d in [
  NVME_CACHE_ROOT,
  HF_HOME_DIR,
  HF_HUB_CACHE_DIR,
  TRANSFORMERS_CACHE_DIR,
  TORCH_HOME_DIR,
  XDG_CACHE_DIR,
]:
  d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HUB_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(TRANSFORMERS_CACHE_DIR)
os.environ['TORCH_HOME'] = str(TORCH_HOME_DIR)
os.environ['XDG_CACHE_HOME'] = str(XDG_CACHE_DIR)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DATA_DIR = PROJECT_ROOT / 'data'
PROMPT_PATH = PROJECT_ROOT / 'prompts' / 'only_pddl_instruction_unraveling.txt'
DOMAIN_PDDL_PATH = DATA_DIR / 'pddl' / 'domain.pddl'
DESC_DIR = DATA_DIR / 'descriptions_unraveling'

OUTPUT_ROOT = PROJECT_ROOT / 'output' / 'direct_pddl_unraveling' / 'qwen2_5_32b_instruct_kani_retry3_unraveling'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path('/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate')

# Model setup (Qwen2.5 Instruct via local HuggingFace + Kani)
MODEL_NAME = 'Qwen2.5-32B-Instruct'
HF_MODEL_ID = f'Qwen/{MODEL_NAME}'

QWEN_ENGINE = HuggingEngine(
  model_id=HF_MODEL_ID,
  model_load_kwargs={
      'device_map': 'auto',
      'cache_dir': str(TRANSFORMERS_CACHE_DIR),
  },
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('VAL exists:', VAL_PATH.exists())
print('Domain PDDL exists:', DOMAIN_PDDL_PATH.exists())
print('Prompt exists:', PROMPT_PATH.exists())
print('Model:', HF_MODEL_ID)
print('NVME cache root:', NVME_CACHE_ROOT)

In [ ]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 101)]  # p01..p100

PROMPT_PREFIX = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

In [ ]:
# Cell 3: Load NL descriptions for one problem ID
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain NL file: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem NL file: {problem_path}")

  return domain_path.read_text(), problem_path.read_text()

In [ ]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  nl_domain, nl_problem = load_problem_descriptions(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer and expert Python engineer.\n"
      "Create an executable Python generator that writes a syntactically correct and semantically consistent "
      "problem file using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY an executable Python generator script.\n"
      "- Wrap it in exactly one <generator>...</generator> block.\n"
      "- The script must write ONLY the final problem PDDL to 'problem.pddl' in current working directory.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not include any extra text before or after <generator>...</generator>."
  )


In [ ]:
# Cell 5: Call Qwen2.5 via Kani for one prompt
async def _call_qwen_async(prompt: str, temperature: float = 0.4, max_new_tokens: int = 32768):
  ai = Kani(QWEN_ENGINE, system_prompt='')
  return await ai.chat_round_str(
      prompt,
      temperature=temperature,
      max_new_tokens=max_new_tokens,
  )


def call_qwen(prompt: str, temperature: float = 0.4, max_output_tokens: int = 32768):
  try:
      loop = asyncio.get_running_loop()
  except RuntimeError:
      return asyncio.run(_call_qwen_async(prompt, temperature=temperature, max_new_tokens=max_output_tokens))

  import nest_asyncio
  nest_asyncio.apply()
  return loop.run_until_complete(_call_qwen_async(prompt, temperature=temperature, max_new_tokens=max_output_tokens))

In [ ]:
# Cell 6: Parse model output and extract generator block
def extract_generator_code(raw_text: str):
  m = re.search(r"<generator>(.*?)</generator>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  return m.group(1).strip() if m else ""

In [ ]:
# Cell 7: Online solver (public endpoint) + raw API JSON persistence
import requests
import re
import json

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()
  res = None

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  # Persist full final API response for debugging/truncation analysis
  (work_dir / "solver_raw_response.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic for zero-cost empty-plan cases
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }